This notebook implements the neural network quantization mechanism from scratch for tensors

In [40]:
import torch

In [41]:
# generate uniformaly distribured random tensor
LOW = -50
HIGH = 150
SIZE = 20
params = (HIGH - LOW) * torch.rand((SIZE, 1)) + LOW

print(params)


tensor([[ 11.7424],
        [ 64.8205],
        [ 26.6250],
        [100.3161],
        [ 16.6246],
        [ 12.8246],
        [128.7953],
        [107.8163],
        [117.6631],
        [115.6776],
        [108.7089],
        [-28.4346],
        [102.4625],
        [ 15.0240],
        [118.0721],
        [ 83.9111],
        [-32.6296],
        [ 67.0219],
        [126.1826],
        [  2.9146]])


In [42]:
params[0] = params.max() + 1
params[1] = params.min() - 1
params[2] = 0

In [43]:
params = torch.round(params, decimals=2)


In [44]:
print(params)

tensor([[129.8000],
        [-33.6300],
        [  0.0000],
        [100.3200],
        [ 16.6200],
        [ 12.8200],
        [128.8000],
        [107.8200],
        [117.6600],
        [115.6800],
        [108.7100],
        [-28.4300],
        [102.4600],
        [ 15.0200],
        [118.0700],
        [ 83.9100],
        [-32.6300],
        [ 67.0200],
        [126.1800],
        [  2.9100]])


In [45]:
def assymetric_quantization(tensor : torch.Tensor, n_bits : int):
    """
    Assymetric quantization of a tensor
    """
    lower_bound = torch.min(tensor)
    upper_bound = torch.max(tensor)
    scale = (upper_bound - lower_bound) / (2 ** n_bits - 1)
    zero = -1.0 * torch.round(lower_bound / scale)
    quantized = torch.clamp(torch.round(tensor/scale + zero), 0, 2**n_bits-1).to(torch.int32)
    return quantized,scale,zero



In [46]:
def symmetric_quantization(tensor, n_bits):
    """
    Symmetric quantization of a tensor
    """
    upper_bound = torch.max(torch.abs(tensor))
    scale = upper_bound / (2**(n_bits-1)-1)
    quantized = torch.clamp(torch.round(tensor/scale), -2**(n_bits-1), 2**(n_bits-1)-1).to(torch.int32)
    return quantized, scale


In [47]:
def asymmetric_dequantization(quantized, scale, zero):
    return (quantized - zero) * scale

In [48]:
def symmetric_dequantization(quantized, scale):
    return quantized * scale

In [49]:
def quantization_error(original, quantized):
    return torch.mean((original-quantized)**2)

In [50]:
assymetric_q, scale_a, zero_a = assymetric_quantization(params, 8)
symmetric_q, scale_s = symmetric_quantization(params, 8)

In [51]:
print(assymetric_q)
print(symmetric_q)

tensor([[255],
        [  0],
        [ 52],
        [209],
        [ 78],
        [ 72],
        [253],
        [220],
        [236],
        [232],
        [222],
        [  8],
        [212],
        [ 75],
        [236],
        [183],
        [  1],
        [157],
        [249],
        [ 57]], dtype=torch.int32)
tensor([[127],
        [-33],
        [  0],
        [ 98],
        [ 16],
        [ 13],
        [126],
        [105],
        [115],
        [113],
        [106],
        [-28],
        [100],
        [ 15],
        [116],
        [ 82],
        [-32],
        [ 66],
        [123],
        [  3]], dtype=torch.int32)


In [52]:
params_deq_a = asymmetric_dequantization(assymetric_q, scale_a, zero_a)
params_deq_s = symmetric_dequantization(symmetric_q, scale_s)

In [53]:
print("symmetric error = ", quantization_error(params, params_deq_s))
print("asymmetric error = ", quantization_error(params, params_deq_a))

symmetric error =  tensor(0.0827)
asymmetric error =  tensor(0.0431)


the input tensor is not perfectly symmetric, it has a range of -50 to 150. hence we see asymmetric quantization giving better results compared to symmetric quantization